In [5]:
import requests
import pandas as pd
import numpy as np
import random
import os
import hashlib
import json
from langchain_deepseek import ChatDeepSeek
from langchain.schema import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import XinferenceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import EvaluationDataset, SingleTurnSample
from ragas import evaluate, metrics
from ragas.run_config import RunConfig
from pydantic import SecretStr
from tqdm import tqdm
from loguru import logger
from pathlib import Path
from datetime import datetime

In [2]:
# 读取 TEST 数据集
logger.info('Loading TEST JSONL file...')
df = pd.read_json("qac_dataset_test.jsonl", lines=True, encoding="utf-8")
logger.success("TEST JSONL file is read.")

# 创建 List 存储 QAC 数据
logger.info('Creating QAC list...')
qac_list = []
for index, row in df.iterrows():
    qac_list.append({
        "question": row["question"],
        "answer": row["answer"],
        "context": row["context"]
    })
logger.success(f"QAC list is created. Total QAC size: {len(qac_list)}")

2025-06-24 00:42:04.584 | INFO     | __main__:<module>:2 - Loading TEST JSONL file...
2025-06-24 00:42:04.592 | SUCCESS  | __main__:<module>:4 - TEST JSONL file is read.
2025-06-24 00:42:04.593 | INFO     | __main__:<module>:7 - Creating QAC list...
2025-06-24 00:42:04.600 | SUCCESS  | __main__:<module>:15 - QAC list is created. Total QAC size: 200


In [6]:
os.environ["DEEPSEEK_API_KEY"] = "sk-574d077e01be45beab39804304a15109"

# 创建 Deepseek langchain Client
chat = ChatDeepSeek(
    model="deepseek-reasoner",
    temperature=0,
    max_tokens=4000,
    timeout=None,
    max_retries=2,
)

# 提交 QAC 数据集
logger.info('Submitting QAC items to LLM...')
samples = []
for qac in tqdm(
    qac_list, desc="Submitting QAC items to LLM",
    unit="item", total=len(qac_list)
):
    # 拿到 QAC 数据
    question = qac.get("question", "")
    reference = qac.get("answer", "")
    reference_contexts = [qac.get("context", "")]

    # 使用 LLM 回答问题
    response = chat.invoke([
        SystemMessage("Please provide responses to the following questions regarding the field of para-hydrogen-induced hyperpolarization."),
        HumanMessage(content=question)
    ]).content

    # 创建 SingleTurnSample 对象
    # user_input: 用户输入的问题
    # retrieved_contexts: AI 召回的相关文本
    # response: AI 生成的回答
    # reference_contexts: 人给出的正确召回片段
    # reference: 人给出的正确回答
    sample = SingleTurnSample(
        user_input=question,
        retrieved_contexts=[],
        response=response,
        reference_contexts=reference_contexts,
        reference=reference
    )

    # 将样本添加到列表
    samples.append(sample)

logger.success('Submitted QAC items to LLM.')

2025-06-24 00:46:36.549 | INFO     | __main__:<module>:13 - Submitting QAC items to LLM...
Submitting QAC items to LLM: 100%|██████████| 200/200 [2:48:14<00:00, 50.47s/item]  
2025-06-24 03:34:50.565 | SUCCESS  | __main__:<module>:47 - Submitted QAC items to LLM.


In [7]:
try:
    logger.info("Creating the LLM and Embeddings.")
    # 创建 LLM 和 Embeddings 模型
    evaluator_llm = LangchainLLMWrapper(
        ChatOpenAI(
            model="deepseek-chat",
            api_key=SecretStr("sk-574d077e01be45beab39804304a15109"),
            base_url="https://api.deepseek.com/v1",
            temperature=0,
            max_tokens=None,
            timeout=None,
            max_retries=2,
        )
    )
    logger.success(f"LLM successfully created.")
    # 用于评估的嵌入模型
    evaluator_embeddings = LangchainEmbeddingsWrapper(
        XinferenceEmbeddings(
            server_url="http://10.26.58.108:9998",
            model_uid="bge-m3-MsvUdbGI"
        )
    )
    logger.success(f"Embeddings successfully created.")
except Exception as e:
    logger.exception(f"Failed to initialize LLM or Embeddings: {e}")
    raise

2025-06-24 12:13:15.001 | INFO     | __main__:<module>:2 - Creating the LLM and Embeddings.
2025-06-24 12:13:15.236 | SUCCESS  | __main__:<module>:15 - LLM successfully created.
2025-06-24 12:13:15.723 | SUCCESS  | __main__:<module>:23 - Embeddings successfully created.


In [8]:
# 将最后的结果添加到 EvaluationDataset 中
eval_dataset = EvaluationDataset(samples=samples)

# 评估指标列表
metrics_list = [
    # 答案正确性，通过嵌入模型比较答案的相似程度
    metrics.answer_correctness,
    # 答案相关性，越是不完整或包含冗余信息的答案，得分越低
    metrics.answer_relevancy,
    # 忠实度/可信度，衡量了生成的答案与给定上下文的事实一致性
    metrics.faithfulness,
    # 上下文精度，评估所有在上下文中呈现的与基本事实相关的条目是否排名较高。
    metrics.context_precision,
    # 上下文召回率，衡量检索到的上下文与人类提供的真实答案的一致程度。
    metrics.context_recall,
]

# 创建 run config
eval_run_config = RunConfig(timeout=600, log_tenacity=True)

# 评估数据集
logger.info('Evaluating QAC items...')
results = evaluate(
    dataset=eval_dataset,
    metrics=metrics_list,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=eval_run_config
)
logger.success('Evaluated QAC items.')

# 保存结果到 CSV 文件
logger.info('Saving results to CSV file...')
results.to_pandas().to_csv("qac_results.csv")
logger.success('Results saved to CSV file.')

2025-06-24 12:13:17.319 | INFO     | __main__:<module>:22 - Evaluating QAC items...


Evaluating:   0%|          | 0/1000 [00:00<?, ?it/s]

Exception raised in Job[76]: OutputParserException(Invalid json output: {"question": "Why does the deviation of the determined rate constant to lower-than-expected values occur when \(R_1 \\leq k_{\\mathrm{obs}}\) in PHIP simulations?", "noncommittal": 0}
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE )
Exception raised in Job[372]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
Exception raised in Job[511]: OutputParserException(Failed to parse StringIO from completion {"question": "How does saturating different Exchangeable Proton (EA) \\(^{1}\\\\text{H}\\) resonances affect PRINOE (Para-hydrogen-Induced Nuclear Overhauser Effect) polarization?", "noncommittal": 0}. Got: 1 validation error for StringIO
text
  Field required [type=missing, input_value={'question': 'How does sa...on?', 'noncommittal': 0}, input_type=dict]
    For further information visit https://